<a href="https://colab.research.google.com/github/dhairye/VRP-DACT/blob/modified-cost-function/Play_with_DACT_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning to Iteratively Solve Routing Problems with DACT

Thank you for your interest in our work!<br>

In this notebook, I will provide some demos on how to run and play with our DACT framework.<br>
Hope this helps with your project :)

For any queries and potential collaboration, please email me at: yiningma@u.nus.edu<br>
The code is free to use for personal and academic usage. For other purposes, please contact me.

The code is based on the following paper:

Yining Ma, Jingwen Li, Zhiguang Cao, Wen Song, Le Zhang, Zhenghua Chen, Jing Tang, “Learning to iteratively solve routing problems with dual-aspect collaborative transformer,” in Advances in Neural Information Processing Systems, vol. 34, 2021.

In [1]:
# !pip install protobuf==3.20.0
!pip install tensorboard_logger
!pip install tqdm

In [2]:
import os
import json
import torch
import pprint
import numpy as np
from tensorboard_logger import Logger as TbLogger
import warnings

from problems.problem_tsp import TSP
from problems.problem_vrp import CVRP
from agent.ppo import PPO
from tqdm.notebook import tqdm

ModuleNotFoundError: No module named 'problems'

## 1. load the settings for training/inference
We first load the settings for training or inference.<br>
In our code, we use the file 'options.py' and the variable 'opts' to store all hyper-parameters.

In [ ]:
from options import get_options
opts = get_options('')
opts

## 2. TSP-20 example

We now try TSP-20 and the '2-opt' decoder. The initial solutions are randomly initialized and we turn off the tensorboard logging and model saving functions. As an example, we train our DACT for 3 epochs (128 instances per epoch) with batch size 32. And we will validate the performance on 10 instances.

In [ ]:
opts.problem = 'tsp'
opts.graph_size = 20
opts.val_dataset='./datasets/tsp_20_10000.pkl'
opts.step_method = '2_opt'
opts.init_val_met = 'random'
opts.no_saving = True
opts.no_tb = True
opts.batch_size = 16
opts.epoch_end = 3
opts.epoch_size = 128
opts.T_max = 1000
opts.val_size = 10
tb_logger = None
opts

In [ ]:
# Optionally configure tensorboard
tb_logger = None
if not opts.no_tb and not opts.distributed:
    tb_logger = TbLogger(os.path.join(opts.log_dir, "{}_{}".format(opts.problem,
                                                      opts.graph_size), opts.run_name))
if not opts.no_saving and not os.path.exists(opts.save_dir):
    os.makedirs(opts.save_dir)

# Save arguments so exact configuration can always be found
if not opts.no_saving:
    with open(os.path.join(opts.save_dir, "args.json"), 'w') as f:
        json.dump(vars(opts), f, indent=True)

# Set the device
opts.device = torch.device("cuda" if opts.use_cuda else "cpu")

We now get our DACT model (agent.actor) and the environment (problem) objects. In our code,

- The problems are defined in the folder 'problems/' for TSP and CVRP

- The model architecture is defined in the folder './nets' where we put some fundamental classes in file 'graph_layers.py' and we define our actor (DACT) and critic in 'actor_network.py' and 'critic_network.py' respectively.
- The PPO class defined in file 'agent/ppo.py' stores all the procedures of training/inference, and here we name the PPO object as the 'agent'. Thus agent.actor is DACT and agent.critic is the critic.

In [ ]:
def load_agent(name):
    agent = {
        'ppo': PPO,
    }.get(name, None)
    assert agent is not None, "Currently unsupported agent: {}!".format(name)
    return agent

def load_problem(name):
    problem = {
        'tsp': TSP,
        'vrp': CVRP,
    }.get(name, None)
    assert problem is not None, "Currently unsupported problem: {}!".format(name)
    return problem

# Figure out what's the problem
problem = load_problem(opts.problem)(
                        p_size = opts.graph_size,
                        step_method = opts.step_method,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        P = opts.P,
                        DUMMY_RATE = opts.dummy_rate)

# Figure out the RL algorithm
agent = load_agent(opts.RL_agent)(problem.NAME, problem.size,  opts)

In [ ]:
print(agent.actor)

In [ ]:
print(agent.critic)

In [ ]:
agent.optimizer

### Test before training
We first test the DACT on 10 instances with T=1k steps to see the performance of random initialization.

As can be seen, the best average costs after 1k steps of search is around 7.414

In [ ]:
agent.start_inference(problem, opts.val_dataset, tb_logger)

### Train for 3 epochs

We now perform a short training (3 epochs) for demonstration. (using CPU on my Macbook, you can use GPU on your machine:))

In [ ]:
 agent.start_training(problem, opts.val_dataset, tb_logger)

### Test after short-training
We then test the DACT on the same 10 instances with T=1k steps to see the performance after the above short training.

As can be seen, the best average cost after 1k steps of search is around 6.123 which shows some imporvement. But off cause such toy training with the small-scale training data is not enough for the Transformer model to get good results. Please follow the hyper-parameters in our paper for actually training.

In [ ]:
agent.start_inference(problem, opts.val_dataset, tb_logger)

### Load the pre-trained model to test
We now load the pre-trained model to see the real performance of our DACT.

In [ ]:
opts.load_path = './pretrained/tsp20-epoch-199.pt'

assert opts.load_path is None or opts.resume is None, "Only one of load path and resume can be given"
load_path = opts.load_path if opts.load_path is not None else opts.resume
if load_path is not None:
    agent.load(load_path)

agent.start_inference(problem, opts.val_dataset, tb_logger)

## 3. CVRP example

We now try CVRP. Different from TSP, we consider the dummy depots to handle the sub-tours in CVRP (see our Appendix in https://arxiv.org/abs/2110.02544 for detailed explanation).

In [ ]:
opts.problem = 'vrp'
opts.step_method = '2_opt'
opts.init_val_met = 'random'
opts.no_saving = True
opts.no_tb = True
opts.T_max = 1000
opts.val_size = 10
tb_logger = None
opts

### Load the pre-trained model to test
We now load the pre-trained model to see the performance.

#### CVRP-20
Note that we cosider 10 dummy depots for CVRP-20. So the dummy rate is 0.5 (0.5*20=10).

In [ ]:
opts.graph_size = 20
opts.dummy_rate = 0.5
opts.val_dataset='./datasets/cvrp_20_10000.pkl'

problem = load_problem(opts.problem)(
                        p_size = opts.graph_size,
                        step_method = opts.step_method,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        P = opts.P,
                        DUMMY_RATE = opts.dummy_rate)

agent = load_agent(opts.RL_agent)(problem.NAME, problem.size,  opts)

opts.load_path = 'pretrained/cvrp20-epoch-191.pt'

assert opts.load_path is None or opts.resume is None, "Only one of load path and resume can be given"
load_path = opts.load_path if opts.load_path is not None else opts.resume
if load_path is not None:
    agent.load(load_path)

agent.start_inference(problem, opts.val_dataset, tb_logger)

#### CVRP-50
Note that we cosider 20 dummy depots for CVRP-50. So the dummy rate is 0.4 (0.4*50=20).

In [ ]:
opts.graph_size = 50
opts.dummy_rate = 0.4
opts.val_dataset='./datasets/cvrp_50_10000.pkl'

problem = load_problem(opts.problem)(
                        p_size = opts.graph_size,
                        step_method = opts.step_method,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        P = opts.P,
                        DUMMY_RATE = opts.dummy_rate)

agent = load_agent(opts.RL_agent)(problem.NAME, problem.size,  opts)

opts.load_path = 'pretrained/cvrp50-epoch-197.pt'

assert opts.load_path is None or opts.resume is None, "Only one of load path and resume can be given"
load_path = opts.load_path if opts.load_path is not None else opts.resume
if load_path is not None:
    agent.load(load_path)

agent.start_inference(problem, opts.val_dataset, tb_logger)

#### CVRP-100
Note that we cosider 20 dummy depots for CVRP-100. So the dummy rate is 0.2 (0.2*100=20).

In [ ]:
opts.graph_size = 100
opts.dummy_rate = 0.2
opts.val_dataset='./datasets/cvrp_100_10000.pkl'

problem = load_problem(opts.problem)(
                        p_size = opts.graph_size,
                        step_method = opts.step_method,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        P = opts.P,
                        DUMMY_RATE = opts.dummy_rate)

agent = load_agent(opts.RL_agent)(problem.NAME, problem.size,  opts)

opts.load_path = 'pretrained/cvrp100-epoch-198.pt'

assert opts.load_path is None or opts.resume is None, "Only one of load path and resume can be given"
load_path = opts.load_path if opts.load_path is not None else opts.resume
if load_path is not None:
    agent.load(load_path)

agent.start_inference(problem, opts.val_dataset, tb_logger)

## 4. Look inside the data structure (Taking TSP-20 as an example)

If you want to modify DACT for your own task, I bet this part is important and helpful for you to understand the current data structure of the DACT :)

### Switch back to TSP-20
Let's first switch back to TSP-20 and load the pre-trained DACT model.

In [ ]:
opts.problem = 'tsp'
opts.graph_size = 20
opts.val_dataset='./datasets/tsp_20_10000.pkl'
opts.step_method = '2_opt'
opts.init_val_met = 'random'
opts.no_saving = True
opts.no_tb = True
opts.T_max = 1000
opts.val_size = 10
tb_logger = None

problem = load_problem(opts.problem)(
                        p_size = opts.graph_size,
                        step_method = opts.step_method,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        P = opts.P,
                        DUMMY_RATE = opts.dummy_rate)

agent = load_agent(opts.RL_agent)(problem.NAME, problem.size,  opts)

opts.load_path = './pretrained/tsp20-epoch-199.pt'

assert opts.load_path is None or opts.resume is None, "Only one of load path and resume can be given"
load_path = opts.load_path if opts.load_path is not None else opts.resume
if load_path is not None:
    agent.load(load_path)


### initilize 10 instances and show the first one

In [ ]:
from torch.utils.data import DataLoader
from problems.problem_tsp import TSPDataset
dataset = TSPDataset(size = 20, num_samples = 10)
batch = next(iter(DataLoader(dataset, batch_size=10)))
coordinates_first = batch['coordinates'][0]
print(coordinates_first)

### Get initial solutions ramdomly

#### **important note !!!!**

**Please note that in our implementation, the solution is stored in a linked list format. Here, if rec[i] = j, it means the node i is connected to node j, i.e., edge i-j is in the solution. For example, if edge 0-1, edge 1-5, edge 2-10 are in the solution, so we have rec[0]=1, rec[1]=5 and rec[2]=10.**

In [ ]:
rec = problem.get_initial_solutions(batch)
print(rec[0])

Let's plot the initial solution for the first instance to illustrate the linked list format.

In [ ]:
import torch
from matplotlib import pyplot as plt
from problems.problem_tsp import get_real_seq

def plot_tour(rec, coordinates, dpi = 300):
    plt.figure(figsize=(8,6))
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.axis([-0.05, 1.05]*2)
    # plot the nodes
    plt.scatter(coordinates[:,0], coordinates[:,1], marker = 'H', s = 55, c = 'blue', zorder = 2)
    # plot the tour
    real_seq = get_real_seq(rec.unsqueeze(0))
    real_seq_coordinates = coordinates.gather(0,real_seq[0].unsqueeze(1).repeat(1,2))
    real_seq_coordinates = torch.cat((real_seq_coordinates, real_seq_coordinates[:1]),0)
    plt.plot(real_seq_coordinates[:,0], real_seq_coordinates[:,1], color = 'black', zorder = 1)
    # mark node
    for i,txt in enumerate(range(rec.size(0))):
        plt.annotate(txt,(coordinates[i,0]+0.01, coordinates[i,1]+0.01),)


    plt.show()

In [ ]:
plot_tour(rec[0], coordinates_first)
print('Linked list format (rec variable):\n', rec[0])
print('\nHere, the linked list format means:')
for i in range(20):
    print(f'edge {i}-{rec[0,i]} is in the solution')
print('\nReal solution after decoding (node visited in sequence):\n', get_real_seq(rec)[0])

### Use pre-trained DACT to solve this mini-batch

The codes here are dapated from the 'agent.rollout' function defined in line 135 of file 'agent/ppo.py'

In [ ]:
# get initial cost
obj = problem.get_costs(batch, rec)
best_solution = rec.clone()

# prepare the features
batch_feature = problem.input_feature_encoding(batch)
solving_state = torch.zeros((batch_feature.size(0),1), device = opts.device).long()
action = None

for t in tqdm(range(2000), disable = opts.no_progress_bar,
              desc = 'rollout', bar_format='{l_bar}{bar:20}{r_bar}{bar:-20b}'):

     # pass through model
    action = agent.actor(problem,
                          batch_feature,
                          rec,
                          action,
                          do_sample = True)[0]

    # state trasition
    rec, rewards, obj, solving_state = problem.step(batch,
                                                    rec,
                                                    action,
                                                    obj,
                                                    solving_state,
                                                    best_solution = best_solution)

    # record informations
    best_solution[rewards > 0] = rec[rewards > 0]
bv = obj[:,-1].reshape(10, 1).min(1)[0]

### show the improved solution of the first instance

In [ ]:
print('best cost:', bv[0])
print('best rec found:', best_solution[0])

In [ ]:
plot_tour(best_solution[0], coordinates_first)
print('Linked list format (rec variable):\n', best_solution[0])
print('\nHere, the linked list format means:')
for i in range(20):
    print(f'edge {i}-{best_solution[0,i]} is in the best solution')
print('\nReal solution after decoding (node visited in sequence):\n', get_real_seq(best_solution)[0])